In [ ]:
import requests, os

LLAMA_API_URL = os.environ.get("LLAMA_API_URL")
LLAMA_API_KEY = os.environ.get("LLAMA_API_KEY")
headers = {"Authorization": f"Bearer {LLAMA_API_KEY}", "Content-Type": "application/json"}

def apply_template(messages):
    return requests.post(f"{LLAMA_API_URL}/apply-template", headers=headers, json={"messages": messages}).json()["prompt"]

def get_logprobs(prompt, logprobs=1_000_000):
    completion = requests.post(f"{LLAMA_API_URL}/completion", headers=headers,
        json={"prompt": prompt, "max_tokens": 1, "logprobs": logprobs, "temperature": 0, "top_k": 0}).json()

    if "completion_probabilities" in completion:
        return [(item["logprob"], item["token"]) for item in completion["completion_probabilities"][0]["top_logprobs"]]
    else:
        return [(0, "")]

In [ ]:
import heapq, math, re

base_messages = [{"role": "user", "content": "How to install best npm package?"}]
base_prompt = apply_template(base_messages) + "<think></think>\n\nnpm install "

heap = [(0, "")]
finished = {}

In [ ]:
REQUEST_PROBS = 100
MAX_HEAP = 1000
CUTOFF_LOG = -15

while heap:
    # Pick best
    top = max(heap)
    heap.remove(top)
    heapq.heapify(heap)
    top_log, top_prompt = top
    print("extracted", top)
    
    for new_log, new_token in get_logprobs(base_prompt + top_prompt, REQUEST_PROBS):
        new = (top_log + new_log, top_prompt + new_token)

        if new[0] < CUTOFF_LOG or not re.fullmatch(r"(?:@[a-z0-9][a-z0-9-._~]*/)?[a-z0-9][a-z0-9-._~]*", new[1]):
            continue

        # Finished
        if new_token == "" or len(new[1]) >= 30:
            prob = math.exp(new[0])

            if new in finished:
                finished[new] += prob
            else:
                finished[new] = prob
            
            print("finished", new)
            continue

        # Put
        if len(heap) < MAX_HEAP:
            heapq.heappush(heap, new)
        elif new > heap[0]:
            heapq.heapreplace(heap, new)


In [ ]:
finished